# Loss Functions

A loss function checks how far the prediction $\hat{y}$ is from the correct answer $y$.

```mermaid
flowchart LR
    P[Prediction y-hat] --> L[Loss Function]
    T[True label y] --> L
    L --> V[Loss value]
```

- Good prediction = low loss
- Bad prediction = high loss
- Training tries to minimize the loss

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

class_names = ["mammal", "bird", "reptile"]

## Low Loss vs High Loss

A bear is a mammal, so its true label is `0`.

The model output before softmax is called **scores** or **logits**.

In [ ]:
criterion = nn.CrossEntropyLoss()
target = torch.tensor([0])  # mammal

correct_scores = torch.tensor([[4.0, 1.0, 0.2]])  # We have high value at the 0th place
wrong_scores = torch.tensor([[0.2, 1.0, 4.0]])

print(f"Correct prediction loss: {criterion(correct_scores, target).item():.4f}")
print(f"Wrong prediction loss:   {criterion(wrong_scores, target).item():.4f}")

Correct prediction loss: 0.0697
Wrong prediction loss:   3.8697


## One-hot Encoding

Each data row has a target class number. One-hot encoding turns each target into a row of zeros and one `1`.

- Mammal `0` -> `[1, 0, 0]`
- Bird `1` -> `[0, 1, 0]`
- Reptile `2` -> `[0, 0, 1]`

This is similar to a pandas dummy-variable table, but PyTorch starts with numbered class labels.

In [3]:
# One target class for each of the 4 data rows
labels = torch.tensor([0, 1, 2, 0])
one_hot_labels = F.one_hot(labels, num_classes=3)

print(f"Labels: {labels}")
print(f"Labels shape: {labels.shape}")
print(f"One-hot targets:\n{one_hot_labels}")
print(f"One-hot shape: {one_hot_labels.shape}")

tensor([[1, 0, 0],
        [0, 1, 0],
        [0, 0, 1]])


The shapes tell us what happened:

```text
labels shape:  [4]     -> 4 target class numbers
one-hot shape: [4, 3]  -> 4 rows, 3 possible classes
```

Each one-hot row is the target for the matching data row.

## Cross-entropy Loss

The usual PyTorch setup is:

- Input: raw logits with shape `[batch, classes]`
- Target: class numbers with shape `[batch]`
- Output: one loss value

For four rows and three classes:

```text
scores:  [4, 3]  -> 3 predicted scores for each row
targets: [4]     -> 1 correct class number for each row
```

> The one-hot table helps us understand the targets, but `CrossEntropyLoss` normally uses the original class numbers. Do not apply softmax first; the loss function handles it internally.

## Putting It Together

Five animals go through the model. The model returns three scores per animal.

In [11]:
torch.manual_seed(42)

# 5 animals, 6 features each
input_data = torch.tensor([
    [-0.4421,  1.5207,  2.0607, -0.3647,  0.4691,  0.0940],
    [-0.9155, -0.0475, -1.3645,  0.6336, -1.9520, -0.3398],
    [ 0.7406,  1.6763, -0.8511,  0.2432,  0.1123, -0.0633],
    [-1.6630, -0.0718, -0.1285,  0.5396, -0.0288, -0.8622],
    [-0.7413,  1.7920, -0.0883, -0.6685,  0.4745, -0.4245],
])

targets = torch.tensor([0, 0, 2, 1, 1])

model = nn.Sequential(
    nn.Linear(6, 4),
    nn.Linear(4, 3),
)

scores = model(input_data)
one_hot_labels = F.one_hot(targets, 3)
loss = criterion(scores, targets)

print(f"Scores shape: {scores.shape}")
print(scores)
print(one_hot_labels)
print(f"Targets shape: {targets.shape}")
print(f"Loss: {loss.item():.4f}")

Scores shape: torch.Size([5, 3])
tensor([[-0.1486, -0.3481,  0.7965],
        [ 0.0644,  0.0389, -0.5037],
        [ 0.3666,  0.0472,  0.1662],
        [-0.0839, -0.3581,  0.0761],
        [-0.2367, -0.1846,  0.4744]], grad_fn=<AddmmBackward0>)
tensor([[1, 0, 0],
        [1, 0, 0],
        [0, 0, 1],
        [0, 1, 0],
        [0, 1, 0]])
Targets shape: torch.Size([5])
Loss: 1.2508


## Quick Look

```python
criterion = nn.CrossEntropyLoss()
scores = model(input_data)  # raw logits
loss = criterion(scores, targets)
```

The next step is backpropagation: update the model to lower this loss.